<a href="https://colab.research.google.com/github/advayc/tencent-auk/blob/main/tencent_auk_fix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Free T4 local audio-editing demo

This notebook replaces the rate-limited public Tencent AuK Space with local, free models running in a Google Colab **T4** runtime. It uses Tencent's public AuK WAVs only as test files; no personal upload or API key is needed.

Run cells **1–3** once, then run **only one** of cells **4–7** at a time. Do not use **Run all**.

> AuK itself cannot run correctly on a free T4: it requires BF16-capable hardware and more memory than a T4 offers. This is an all-local fallback. The T4 runs Whisper and speaker analysis. Qwen TTS is deliberately loaded in FP32 because FP16 generation is unstable on Turing GPUs such as a T4. The tone cell is controlled re-synthesis in a selectable narrator voice, rather than exact same-voice AuK emotion editing.

## 1. Confirm the free T4 GPU

In Colab choose **Runtime → Change runtime type → T4 GPU → Save**, then run this cell.

In [1]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU found. In Colab select Runtime → Change runtime type → T4 GPU, then reconnect.')

gpu = torch.cuda.get_device_name(0)
memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'Using: {gpu} ({memory_gb:.1f} GB VRAM)')
print('This notebook uses FP16-compatible local models; a free T4 is supported.')

Using: Tesla T4 (14.6 GB VRAM)
This notebook uses FP16-compatible local models; a free T4 is supported.


## 2. Install the local tools

This may take a few minutes. It installs local inference libraries; it does **not** connect to the Tencent Gradio app.

In [2]:
%pip -q install -U qwen-tts faster-whisper
%pip -q install -U librosa soundfile noisereduce
%pip -q install -U speechbrain silero-vad scikit-learn
import importlib.util
assert importlib.util.find_spec('faster_whisper') is not None, 'faster-whisper did not install. Rerun this cell once.'
print('Install complete. Continue to cell 3.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 63.3 MB/s eta 0:00:00
   ━━━━

## 3. Download test WAVs and load shared helpers

The small WAV files below are Tencent's official public AuK examples. They are used only inside this Colab session.

In [3]:
from pathlib import Path
import gc
import importlib.util
import re
import shutil
import subprocess
import time
import sys

import librosa
import numpy as np
import soundfile as sf
import torch
from IPython.display import Audio, display

%cd /content
if not Path('/content/AuK/.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Tencent-Hunyuan/AuK.git', '/content/AuK'], check=True)

WORK = Path('/content/local_audio_demo')
WORK.mkdir(exist_ok=True)
AUK = Path('/content/AuK/assets/demo-input-audio')
DEMOS = {
    'transcript': AUK / 'content-edit/content.wav',
    'emotion': AUK / 'emotion-edit/en-1-input.wav',
    'cleanup': AUK / 'se/se-zh-1-input.wav',
    'speakers': AUK / 'ss/en-1-input.wav',
}
for name, path in DEMOS.items():
    assert path.exists(), f'Missing demo: {path}'
    print(f'{name:11} → {path.name}')

# Repair a partial package install from a previous Colab cell, if necessary.
if importlib.util.find_spec('faster_whisper') is None:
    print('Installing the missing faster-whisper package…')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'faster-whisper'])

ASR = None

def get_asr():
    global ASR
    if ASR is None:
        from faster_whisper import WhisperModel
        try:
            ASR = WhisperModel('base.en', device='cuda', compute_type='float16')
            print('Loaded local Whisper ASR on the T4.')
        except Exception as error:
            print(f'GPU ASR unavailable ({error}); using local CPU ASR instead.')
            ASR = WhisperModel('base.en', device='cpu', compute_type='int8')
    return ASR

def transcribe_with_words(audio_path):
    segments, _ = get_asr().transcribe(str(audio_path), word_timestamps=True, vad_filter=True)
    segments = list(segments)
    text = ' '.join(segment.text.strip() for segment in segments).strip()
    words = []
    for segment in segments:
        for word in (segment.words or []):
            words.append({'word': word.word, 'start': word.start, 'end': word.end})
    return text, words

def norm_tokens(text):
    return re.findall(r"[a-z0-9']+", text.lower())

def find_phrase_span(words, phrase):
    targets = norm_tokens(phrase)
    spoken = [norm_tokens(item['word'])[0] if norm_tokens(item['word']) else '' for item in words]
    for index in range(len(spoken) - len(targets) + 1):
        if spoken[index:index + len(targets)] == targets:
            return words[index]['start'], words[index + len(targets) - 1]['end']
    transcript = ' '.join(item['word'] for item in words)
    raise ValueError(f'Could not locate the exact old phrase. Whisper heard: {transcript}')

def unload_tts():
    globals().pop('tts_model', None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def patch_optional_chat_templates_404():
    # Transformers 4.57 can treat Qwen's absent *optional* template directory as fatal.
    # It is not a missing model file, so safely treat it as an empty directory.
    import transformers.tokenization_utils_base as tokenization_utils_base
    import transformers.utils.hub as transformers_hub
    from huggingface_hub.errors import RemoteEntryNotFoundError
    if getattr(transformers_hub, '_auk_template_404_patch', False):
        return
    original = transformers_hub.list_repo_templates
    def safe_list_repo_templates(*args, **kwargs):
        try:
            return original(*args, **kwargs)
        except RemoteEntryNotFoundError as error:
            if 'additional_chat_templates' in str(error):
                return []
            raise
    transformers_hub.list_repo_templates = safe_list_repo_templates
    tokenization_utils_base.list_repo_templates = safe_list_repo_templates
    transformers_hub._auk_template_404_patch = True

def load_tts(model_id):
    global tts_model
    if importlib.util.find_spec('qwen_tts') is None:
        print('Installing the missing qwen-tts package…')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'qwen-tts'])
    unload_tts()
    patch_optional_chat_templates_404()
    from qwen_tts import Qwen3TTSModel
    # T4 is a Turing GPU: Qwen TTS FP16 generation can trigger a CUDA assertion.
    # FP32 avoids that known numerical issue and fits comfortably in the free T4's 16 GB VRAM.
    kwargs = {'device_map': 'cuda:0', 'dtype': torch.float32}
    tts_model = Qwen3TTSModel.from_pretrained(model_id, **kwargs)
    return tts_model

def show_file(path, title):
    print(f'{title}: {path}')
    display(Audio(filename=str(path)))

print('Shared setup complete. Now run one feature cell (4, 5, 6, or 7).')

/content
transcript  → content.wav
emotion     → en-1-input.wav
cleanup     → se-zh-1-input.wav
speakers    → en-1-input.wav
Shared setup complete. Now run one feature cell (4, 5, 6, or 7).


In [6]:
def patch_optional_chat_templates_404():
    import transformers.tokenization_utils_base as tokenizer_base
    import transformers.utils.hub as transformers_hub

    if getattr(transformers_hub, "_auk_template_404_patch", False):
        return

    original = transformers_hub.list_repo_templates

    def safe_list_repo_templates(*args, **kwargs):
        try:
            return original(*args, **kwargs)
        except Exception as error:
            if (
                error.__class__.__name__ == "RemoteEntryNotFoundError"
                and "additional_chat_templates" in str(error)
            ):
                return []
            raise

    transformers_hub.list_repo_templates = safe_list_repo_templates
    tokenizer_base.list_repo_templates = safe_list_repo_templates
    transformers_hub._auk_template_404_patch = True

print("Compatibility patch applied. Rerun cell 4.")

Compatibility patch applied. Rerun cell 4.


## 4. Transcript edit (replace words while keeping the voice)

whisper finds the old phrase, Qwen TTS voice-clones a replacement phrase, and the notebook crossfades it into the original test file. The supplied defaults are Tencent's official AuK content-edit example.

In [7]:
old_words = 'but accepting what we cannot have'
new_words = 'and living well with dreams unmet'
input_audio = DEMOS['transcript']
output_path = WORK / '01_transcript_edit_local.wav'

full_text, word_times = transcribe_with_words(input_audio)
start_s, end_s = find_phrase_span(word_times, old_words)
print(f'Whisper transcript: {full_text}')
print(f'Replacing {old_words!r} from {start_s:.2f}s to {end_s:.2f}s')

# Take a short non-overlapping reference from the original speaker.
source_24k, source_sr = librosa.load(str(input_audio), sr=24000, mono=True)
ref_end_s = max(2.5, start_s - 0.10)
ref_end = min(len(source_24k), int(ref_end_s * source_sr))
ref_start = max(0, ref_end - int(3.0 * source_sr))
if ref_end - ref_start < int(1.5 * source_sr):
    ref_start = min(len(source_24k) - int(3.0 * source_sr), int((end_s + 0.10) * source_sr))
    ref_start = max(0, ref_start)
    ref_end = min(len(source_24k), ref_start + int(3.0 * source_sr))
reference_path = WORK / 'reference_for_clone.wav'
sf.write(reference_path, source_24k[ref_start:ref_end], source_sr)
reference_text, _ = transcribe_with_words(reference_path)
print(f'Voice-clone reference text: {reference_text}')

tts = load_tts('Qwen/Qwen3-TTS-12Hz-0.6B-Base')
new_wavs, generated_sr = tts.generate_voice_clone(
    text=new_words,
    language='English',
    ref_audio=str(reference_path),
    ref_text=reference_text,
)
replacement = np.asarray(new_wavs[0], dtype=np.float32).squeeze()
unload_tts()

# Match the original phrase duration, then make short equal-power fades at both joins.
a, b = int(start_s * generated_sr), int(end_s * generated_sr)
source, _ = librosa.load(str(input_audio), sr=generated_sr, mono=True)
target_len = max(1, b - a)
replacement = librosa.effects.time_stretch(replacement, rate=len(replacement) / target_len)
replacement = librosa.util.fix_length(replacement, size=target_len)
edited = source.copy()
fade = min(int(0.045 * generated_sr), target_len // 4)
if fade > 1:
    ramp = np.linspace(0, 1, fade, dtype=np.float32)
    edited[a:a + fade] = source[a:a + fade] * (1 - ramp) + replacement[:fade] * ramp
    edited[a + fade:b - fade] = replacement[fade:-fade]
    edited[b - fade:b] = replacement[-fade:] * (1 - ramp) + source[b - fade:b] * ramp
else:
    edited[a:b] = replacement
sf.write(output_path, edited, generated_sr)
show_file(input_audio, 'Original')
show_file(output_path, 'Edited result')
print('Done. Listen for the new phrase near the end of the result.')

Whisper transcript: Do you think the biggest victory is not fighting for what we want, but accepting what we cannot have?
Replacing 'but accepting what we cannot have' from 4.06s to 6.72s
Voice-clone reference text: biggest victory is not fighting for what we want.



    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    



********
********
 


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.83G [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
/usr/local/lib/python3.13/dist-packages/librosa/effects.py:448: FutureWarning: The `hop_length` parameter is deprecated as of 1.0 and will be removed in 1.1. It is unused in the current implementation.
  stft_stretch = core.phase_vocoder(
/usr/local/lib/python3.13/dist-packages/librosa/effects.py:448: FutureWarning: The `n_fft` parameter is deprecated as of 1.0 and will be removed in 1.1. It is unused in the current implementation.
  stft_stretch = core.phase_vocoder(


Original: /content/AuK/assets/demo-input-audio/content-edit/content.wav


Edited result: /content/local_audio_demo/01_transcript_edit_local.wav


Done. Listen for the new phrase near the end of the result.


## 5. Tone/emotion edit — excited, calm, etc.

This transcribes the supplied English demo and re-synthesizes the same words locally in a Qwen narrator voice following your emotion instruction. It is intentionally labelled re-synthesis because the free T4 path does not preserve the original person's exact identity.

In [ ]:
emotion = 'excited'  # Try: calm, cheerful, serious, warm, anxious
speaker = 'Ryan'     # English voices: Ryan or Aiden
input_audio = DEMOS['emotion']
output_path = WORK / '02_emotion_resynthesis_local.wav'

spoken_text, _ = transcribe_with_words(input_audio)
print(f'Whisper transcript: {spoken_text}')
instruction = f'Speak in a clearly {emotion} tone while remaining natural and easy to understand.'

tts = load_tts('Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice')
wavs, sample_rate = tts.generate_custom_voice(
    text=spoken_text,
    language='English',
    speaker=speaker,
    instruct=instruction,
)
sf.write(output_path, np.asarray(wavs[0], dtype=np.float32).squeeze(), sample_rate)
unload_tts()
show_file(input_audio, 'Original')
show_file(output_path, f'{emotion.title()} local re-synthesis')
print(f'Done. This is an {emotion} delivery in Qwen’s {speaker} narrator voice.')

## 6. Speech cleanup — reduce noise and improve clarity

This local spectral cleanup runs without the DeepFilterNet build dependency. The supplied sample is Tencent's noisy/reverberant cleanup test WAV. It primarily reduces background noise; room reverb can be softened but cannot be perfectly removed by this lightweight free fallback.

In [ ]:
input_audio = DEMOS['cleanup']
output_path = WORK / '03_speech_cleanup_local.wav'

import noisereduce as nr
from scipy.signal import butter, sosfiltfilt

audio, sample_rate = librosa.load(str(input_audio), sr=None, mono=True)
# Remove low-frequency rumble, then apply non-stationary spectral noise reduction.
highpass = butter(4, 80, btype='highpass', fs=sample_rate, output='sos')
audio = sosfiltfilt(highpass, audio).astype(np.float32)
enhanced = nr.reduce_noise(y=audio, sr=sample_rate, stationary=False, prop_decrease=0.85)
sf.write(output_path, enhanced, sample_rate)
show_file(input_audio, 'Original noisy/reverberant audio')
show_file(output_path, 'Local cleanup result')
print('Done. Local spectral cleanup reduced background noise and rumble.')

## 7. Speaker isolation — keep one speaker and mute the others

This local approach detects speech turns, groups similar voices, then preserves the selected speaker's turns. It works best for turn-taking dialogue. When two people speak at exactly the same time, the other person may remain audible because this is diarization-and-muting rather than magical unmixing.

In [ ]:
target_speaker = 1   # 1 = first distinct voice to speak; change to 2 for the other speaker
expected_speakers = 2
input_audio = DEMOS['speakers']
output_path = WORK / f'04_speaker_{target_speaker}_isolated_local.wav'

from silero_vad import get_speech_timestamps, load_silero_vad
from speechbrain.inference.speaker import SpeakerRecognition
from sklearn.cluster import KMeans

sample_rate = 16000
wave, _ = librosa.load(str(input_audio), sr=sample_rate, mono=True)
vad = load_silero_vad()
turns = get_speech_timestamps(
    torch.from_numpy(wave), vad, sampling_rate=sample_rate, return_seconds=True,
    min_speech_duration_ms=250, min_silence_duration_ms=180, speech_pad_ms=60,
)

# Break unusually long turns into chunks that can be compared by voice identity.
chunks = []
for turn in turns:
    start, end = float(turn['start']), float(turn['end'])
    cursor = start
    while cursor < end:
        next_cursor = min(end, cursor + 2.0)
        if next_cursor - cursor >= 0.45:
            chunks.append((cursor, next_cursor))
        cursor = next_cursor

if len(chunks) < expected_speakers:
    raise RuntimeError(f'Only {len(chunks)} usable speech chunks were detected; try an audio file with clearer turn-taking.')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
embedder = SpeakerRecognition.from_hparams(
    source='speechbrain/spkrec-ecapa-voxceleb',
    savedir='/content/models/ecapa-voxceleb',
    run_opts={'device': device},
)
embeddings = []
for start, end in chunks:
    segment = wave[int(start * sample_rate):int(end * sample_rate)]
    if len(segment) < sample_rate // 2:
        segment = np.pad(segment, (0, sample_rate // 2 - len(segment)))
    tensor = torch.from_numpy(segment.astype(np.float32)).unsqueeze(0)
    with torch.inference_mode():
        embedding = embedder.encode_batch(tensor).squeeze().cpu().numpy()
    embeddings.append(embedding)

cluster_ids = KMeans(n_clusters=expected_speakers, random_state=42, n_init=10).fit_predict(np.vstack(embeddings))
cluster_order = sorted(set(cluster_ids), key=lambda cluster: min(chunks[i][0] for i, label in enumerate(cluster_ids) if label == cluster))
if not 1 <= target_speaker <= len(cluster_order):
    raise ValueError(f'target_speaker must be from 1 to {len(cluster_order)}')
target_cluster = cluster_order[target_speaker - 1]

mask = np.zeros_like(wave, dtype=np.float32)
fade_samples = int(0.035 * sample_rate)
for (start, end), cluster in zip(chunks, cluster_ids):
    speaker_number = cluster_order.index(cluster) + 1
    print(f'{start:5.2f}s–{end:5.2f}s  → speaker {speaker_number}')
    if cluster == target_cluster:
        a, b = int(start * sample_rate), int(end * sample_rate)
        window = np.ones(b - a, dtype=np.float32)
        fade = min(fade_samples, len(window) // 2)
        if fade > 1:
            window[:fade] = np.linspace(0, 1, fade, dtype=np.float32)
            window[-fade:] = np.linspace(1, 0, fade, dtype=np.float32)
        mask[a:b] = np.maximum(mask[a:b], window)

isolated = wave * mask
sf.write(output_path, isolated, sample_rate)
show_file(input_audio, 'Original multi-speaker audio')
show_file(output_path, f'Isolated speaker {target_speaker}')
print('Done. Change target_speaker to 2 and rerun this cell to hear the other voice.')

## Optional: download your results

Run this after at least one feature cell has finished.

In [ ]:
from google.colab import files
import zipfile

zip_path = WORK / 'local_audio_demo_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for wav in WORK.glob('*.wav'):
        if wav.name != 'reference_for_clone.wav':
            archive.write(wav, wav.name)
files.download(str(zip_path))